# 03 · Criterios de inclusión y dataset analítico

Se aplican los criterios en orden y se registra para cada municipio PROFECO **por qué**
aparece o no en la gráfica.

| # | Criterio | Umbral y justificación |
|---|---|---|
| 1 | Correspondencia geográfica | directa o aproximada/documentada (notebook 01) |
| 2 | Establecimientos de autoservicio en el área | ≥ 3 (con 1–2 la mediana depende de una tienda) |
| 3 | Productos de la canasta | 100 % (no se imputan faltantes) |
| 4 | Productos con un solo establecimiento | ≤ 2 de 21 (≈10 %); la distribución se muestra abajo |
| 5 | Precisión ENSU del % «seguro» | CV < 30 % (precisión alta o moderada según INEGI) |

La ventana temporal (precios de agosto vs. ENSU de junio) es la misma para todas las
áreas, por eso se documenta como limitación y no como criterio por área.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
PROC = ROOT / "data/processed"
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
params = json.loads((ROOT / "data/metadata/canasta_parametros.json").read_text(encoding="utf-8"))
items = params["productos"]
homolog = pd.read_csv(PROC / "homologacion_geografica.csv", dtype={"cd_ensu": str, "cve_mun": str, "cve_ent": str})
prices = pd.read_csv(PROC / "precios_area_producto.csv", dtype={"cd_ensu": str})
obs = pd.read_csv(PROC / "observaciones_canasta.csv", dtype={"cd_ensu": str}, parse_dates=["fecha"])
ensu = pd.read_csv(PROC / "ensu_percepcion_areas.csv")
from src.basket import basket_cost
print(len(items), "productos;", obs.area_urbana_ensu.nunique(), "áreas con observaciones de autoservicio")

21 productos; 52 áreas con observaciones de autoservicio


In [2]:
basket = basket_cost(prices, items)
p = prices[prices.producto_normalizado.isin(items)]
single = p[p.establecimientos == 1].groupby("area_urbana_ensu").size().rename("productos_un_establecimiento")
area = (obs.groupby("area_urbana_ensu").agg(establecimientos=("establecimiento_id", "nunique"),
            observaciones_precio=("precio_unitario", "size"))
        .join(obs[obs.producto_normalizado.isin(items)].groupby("area_urbana_ensu").precio_unitario.size()
              .rename("observaciones_canasta"))
        .join(obs.drop_duplicates("establecimiento_id").groupby("area_urbana_ensu")[["lat", "lon"]].median())
        .join(single).fillna({"productos_un_establecimiento": 0}).reset_index())
area = area.merge(basket.drop(columns="cd_ensu"), on="area_urbana_ensu", how="left")
display(area.productos_un_establecimiento.value_counts().sort_index().to_frame("áreas"))

,áreas
productos_un_establecimiento,
0.0,41
1.0,5
3.0,1
4.0,1
5.0,1
6.0,1
11.0,1
19.0,1


In [3]:
MIN_EST, MAX_SINGLE, MAX_CV = params["min_establecimientos"], 2, 30
geo = homolog[homolog.incluir].groupby("area_urbana_ensu").agg(
    cd_ensu=("cd_ensu", "first"), estado=("estado", lambda s: " / ".join(sorted(set(s)))),
    municipios_profeco=("municipio_profeco", lambda s: ", ".join(s)),
    tipo_correspondencia=("tipo_correspondencia", "first"),
    cobertura_geografica_pct=("cobertura_profeco_area_pct", "first")).reset_index()
cand = geo.merge(area, on="area_urbana_ensu", how="left").merge(
    ensu.drop(columns=["estado_ensu", "region_ensu"]), on="area_urbana_ensu", how="left")

def motivo(r):
    if pd.isna(r.establecimientos):
        return "Sin establecimientos de autoservicio en PROFECO"
    if r.establecimientos < MIN_EST:
        return f"Menos de {MIN_EST} establecimientos de autoservicio ({int(r.establecimientos)})"
    if r.productos_disponibles < len(items):
        return f"Canasta incompleta ({int(r.productos_disponibles)} de {len(items)} productos)"
    if r.productos_un_establecimiento > MAX_SINGLE:
        return f"{int(r.productos_un_establecimiento)} productos con precio de un solo establecimiento"
    if pd.isna(r.seguridad_pct) or r.cv_pct_seguro >= MAX_CV:
        return "Estimación ENSU con precisión baja"
    return ""
cand["motivo_exclusion"] = cand.apply(motivo, axis=1)
cand["pasa_validacion"] = cand.motivo_exclusion.eq("")
display(cand.motivo_exclusion.replace("", "INCLUIDA").value_counts())
display(cand.loc[~cand.pasa_validacion, ["area_urbana_ensu", "municipios_profeco", "establecimientos",
        "productos_disponibles", "productos_un_establecimiento", "motivo_exclusion"]])

motivo_exclusion
INCLUIDA                                             44
Menos de 3 establecimientos de autoservicio (2)       5
Sin establecimientos de autoservicio en PROFECO       3
Menos de 3 establecimientos de autoservicio (1)       2
5 productos con precio de un solo establecimiento     1
Name: count, dtype: int64

,area_urbana_ensu,municipios_profeco,establecimientos,productos_disponibles,productos_un_establecimiento,motivo_exclusion
6,Chihuahua,Chihuahua,5.0,21.0,5.0,5 productos con precio de un solo establecimiento
15,General Escobedo,General Escobedo,NaN,NaN,NaN,Sin establecimientos de autoservicio en PROFECO
17,Guadalupe,Guadalupe,NaN,NaN,NaN,Sin establecimientos de autoservicio en PROFECO
21,Iztacalco,Iztacalco,2.0,20.0,6.0,Menos de 3 establecimientos de autoservicio (2)
24,La Magdalena Contreras,Magdalena Contreras,1.0,19.0,19.0,Menos de 3 establecimientos de autoservicio (1)
33,Pachuca de Soto,Pachuca de Soto,1.0,11.0,11.0,Menos de 3 establecimientos de autoservicio (1)
39,San Nicolás de los Garza,San Nicolás de los Garza,2.0,21.0,3.0,Menos de 3 establecimientos de autoservicio (2)
40,San Pedro Tlaquepaque,Tlaquepaque,NaN,NaN,NaN,Sin establecimientos de autoservicio en PROFECO
46,Tonalá,Tonalá,2.0,21.0,0.0,Menos de 3 establecimientos de autoservicio (2)
51,Xochimilco,Xochimilco,2.0,20.0,4.0,Menos de 3 establecimientos de autoservicio (2)


## Tablas exigidas por la metodología

In [4]:
fi, ff = params["fecha_inicio"], params["fecha_fin"]
canasta = cand[["estado", "municipios_profeco", "area_urbana_ensu", "productos_disponibles", "productos_canasta",
                "cobertura_pct", "establecimientos", "costo_canasta"]].rename(columns={"municipios_profeco": "municipio"})
canasta = canasta.assign(fecha_inicio=fi, fecha_fin=ff).sort_values("costo_canasta")
canasta.to_csv(PROC / "canasta_municipio.csv", index=False, encoding="utf-8-sig")

aud = homolog.merge(cand[["area_urbana_ensu", "establecimientos", "cobertura_pct", "seguridad_pct",
                          "pasa_validacion", "motivo_exclusion"]], on="area_urbana_ensu", how="left")
aud["motivo_exclusion"] = np.where(~aud.incluir, "Sin correspondencia geográfica válida: " + aud.observaciones,
                                   aud.motivo_exclusion)
aud["pasa_validacion"] = aud.incluir & aud.pasa_validacion.fillna(False).astype(bool)
auditoria = aud.rename(columns={"municipio_profeco": "municipio", "establecimientos": "establecimientos_autoservicio_area",
    "cobertura_pct": "cobertura_canasta_pct", "area_urbana_ensu": "area_ensu", "seguridad_pct": "dato_ensu"})[
    ["municipio", "estado", "registros_profeco", "establecimientos_profeco", "establecimientos_autoservicio_area",
     "cobertura_canasta_pct", "area_ensu", "tipo_correspondencia", "dato_ensu", "pasa_validacion", "motivo_exclusion"]]
auditoria.to_csv(PROC / "auditoria_cobertura.csv", index=False, encoding="utf-8-sig")
display(auditoria.pasa_validacion.value_counts())

final = cand[cand.pasa_validacion].assign(
    fecha_inicio_precios=fi, fecha_fin_precios=ff, periodo_ensu="junio 2026",
    precision_ensu=lambda d: np.where(d.cv_pct_seguro < 15, "alta", "moderada"))
final = final.rename(columns={"municipios_profeco": "municipio", "lat": "latitud", "lon": "longitud"})[
    ["estado", "municipio", "area_urbana_ensu", "cd_ensu", "tipo_correspondencia", "cobertura_geografica_pct",
     "costo_canasta", "seguridad_pct", "inseguridad_pct", "no_responde_pct", "cv_pct_seguro", "ic90_inf_seguro",
     "ic90_sup_seguro", "precision_ensu", "productos_canasta", "cobertura_pct", "establecimientos",
     "observaciones_precio", "observaciones_canasta", "productos_un_establecimiento", "fecha_inicio_precios",
     "fecha_fin_precios", "periodo_ensu", "latitud", "longitud"]].sort_values("costo_canasta")
final.to_csv(PROC / "dataset_final.csv", index=False, encoding="utf-8-sig")
display(final.round(1))

pasa_validacion
True     52
False    20
Name: count, dtype: int64

,estado,municipio,area_urbana_ensu,cd_ensu,tipo_correspondencia,cobertura_geografica_pct,costo_canasta,seguridad_pct,inseguridad_pct,no_responde_pct,cv_pct_seguro,ic90_inf_seguro,ic90_sup_seguro,precision_ensu,productos_canasta,cobertura_pct,establecimientos,observaciones_precio,observaciones_canasta,productos_un_establecimiento,fecha_inicio_precios,fecha_fin_precios,periodo_ensu,latitud,longitud
7,Estado de México,Nezahualcóyotl,Ciudad Nezahualcóyotl,28,directa,100.0,944.3,57.9,41.3,0.8,7.2,51.1,64.8,alta,21.0,100.0,4.0,370.0,362.0,0.0,2026-08-17,2026-08-31,junio 2026,19.5,-99.1
25,Baja California Sur,La Paz,La Paz,04,directa,100.0,967.2,60.3,39.0,0.7,5.4,54.9,65.6,alta,21.0,100.0,5.0,469.0,458.0,0.0,2026-08-17,2026-08-31,junio 2026,24.1,-110.3
14,Estado de México,Ecatepec de Morelos,Ecatepec de Morelos,27,directa,100.0,973.3,21.3,78.7,0.0,10.9,17.5,25.1,alta,21.0,100.0,10.0,709.0,697.0,0.0,2026-08-17,2026-08-31,junio 2026,19.5,-99.0
1,Aguascalientes,Aguascalientes,Aguascalientes,01,aproximada/documentada,89.5,981.3,59.9,39.8,0.3,6.1,53.9,65.9,alta,21.0,100.0,5.0,493.0,483.0,0.0,2026-08-17,2026-08-31,junio 2026,21.9,-102.3
0,Guerrero,Acapulco de Juárez,Acapulco de Juárez,21,directa,100.0,983.6,21.1,78.9,0.0,12.8,16.6,25.5,alta,21.0,100.0,5.0,546.0,535.0,0.0,2026-08-17,2026-08-31,junio 2026,16.9,-99.9
37,Campeche,Campeche,San Francisco de Campeche,05,directa,100.0,985.7,33.4,65.9,0.7,9.1,28.4,38.4,alta,21.0,100.0,5.0,490.0,479.0,0.0,2026-08-17,2026-08-31,junio 2026,19.8,-90.5
2,Estado de México,"Atizapán de Zaragoza, Atizapán",Atizapán de Zaragoza,90,directa,100.0,986.7,54.5,45.5,0.0,6.6,48.5,60.4,alta,21.0,100.0,5.0,300.0,297.0,0.0,2026-08-17,2026-08-31,junio 2026,19.6,-99.2
42,Baja California,Tijuana,Tijuana,03,directa,100.0,988.4,32.4,67.6,-0.0,10.3,26.9,37.9,alta,21.0,100.0,7.0,507.0,498.0,0.0,2026-08-17,2026-08-31,junio 2026,32.5,-117.0
11,Morelos,Cuernavaca,Cuernavaca,32,aproximada/documentada,49.2,989.7,20.2,79.8,0.0,14.7,15.3,25.1,alta,21.0,100.0,5.0,417.0,417.0,0.0,2026-08-17,2026-08-31,junio 2026,18.9,-99.2
22,Ciudad de México,Iztapalapa,Iztapalapa,76,directa,100.0,991.1,26.7,73.3,0.0,11.9,21.5,32.0,alta,21.0,100.0,8.0,579.0,569.0,0.0,2026-08-17,2026-08-31,junio 2026,19.4,-99.1


In [5]:
# Controles del dataset final
assert final.area_urbana_ensu.is_unique and final.cd_ensu.is_unique
assert final.cobertura_pct.eq(100).all() and final.costo_canasta.notna().all()
assert final.establecimientos.ge(MIN_EST).all() and final.cv_pct_seguro.lt(MAX_CV).all()
assert final.seguridad_pct.between(0, 100).all() and final.latitud.between(14, 33).all()
assert auditoria.pasa_validacion.sum() == homolog.merge(final[["area_urbana_ensu"]]).shape[0]
display(Markdown(f"**Dataset final:** {len(final)} áreas urbanas comparables, de "
    f"{homolog.loc[homolog.incluir, 'cd_ensu'].nunique()} con correspondencia geográfica. "
    f"Precios del {fi} al {ff}; ENSU junio 2026. "
    f"{(final.precision_ensu == 'moderada').sum()} áreas tienen precisión ENSU moderada (CV 15–30 %)."))

**Dataset final:** 44 áreas urbanas comparables, de 55 con correspondencia geográfica. Precios del 2026-08-17 al 2026-08-31; ENSU junio 2026. 4 áreas tienen precisión ENSU moderada (CV 15–30 %).